# Notebook 01: Databricks Fundamentals for Data Engineering
**Author:** Sathiya Murthy  
**Blog:** Data Engineering Diaries  

## Overview
This notebook provides hands-on exercises for mastering Databricks runtime features:
1. **DBUtils Utilities** (`dbutils.fs`, `dbutils.widgets`, `dbutils.secrets`)
2. **PySpark & Delta Lake Operations** (ACID transactions, time travel, Z-Ordering)
3. **Unity Catalog Governance** (3-tier namespaces, schemas, tables)
4. **Performance Tuning** (Photon, Adaptive Query Execution, Disk Caching)
5. **MLflow Pipeline Tracking** (Logging data quality metrics and runs)

--- 
## Module 1: Databricks DBUtils Simulation & Widget Creation

In [ ]:
# Inspect DBUtils File System Operations
import sys
sys.path.append('..')
from src.dbutils_patterns import DBUtilsPatterns

dbutils_mgr = DBUtilsPatterns()
print("=== Available DBUtils Commands ===")
for cmd in dbutils_mgr.list_fs_commands():
    print(f"  dbutils.fs.{cmd}")

# Create notebook widgets for runtime parameter passing
dbutils_mgr.widgets.text("environment", "dev", "Target Environment")
dbutils_mgr.widgets.dropdown("batch_size", "1000", ["1000", "5000", "10000"], "Batch Size")
print("Selected Environment:", dbutils_mgr.widgets.get("environment"))

--- 
## Module 2: Unity Catalog 3-Tier Namespace Setup

In [ ]:
from src.unity_catalog import UnityCatalogManager

uc = UnityCatalogManager()
ddl_catalog = uc.create_catalog("dev_lakehouse", comment="Dev Lakehouse Metastore Catalog")
ddl_schema = uc.create_schema("dev_lakehouse", "bronze_landing", comment="Raw Landing Zone")
ddl_grant = uc.grant_privilege("data_engineers", "ALL PRIVILEGES", "SCHEMA", "dev_lakehouse.bronze_landing")

print(ddl_catalog)
print(ddl_schema)
print(ddl_grant)

--- 
## Module 3: Auto Loader & Incremental Ingestion Setup

In [ ]:
from src.auto_loader import AutoLoaderIngestion

al = AutoLoaderIngestion(
    file_format="json",
    schema_location="/tmp/autoloader_demo/_schema",
    checkpoint_location="/tmp/autoloader_demo/_checkpoints"
)

pyspark_streaming_script = al.build_pyspark_stream_code(
    source_path="s3://company-landing-bucket/raw_logs/",
    target_delta_table="dev_lakehouse.bronze_landing.raw_logs",
    trigger_mode="availableNow"
)

print("=== Generated Auto Loader Streaming Code ===")
print(pyspark_streaming_script)

--- 
## Module 4: Performance Optimization & AQE Configurations

In [ ]:
from src.performance_tuning import DatabricksPerformanceOptimizer

optimizer = DatabricksPerformanceOptimizer()
configs = optimizer.get_optimized_spark_configs(enable_photon=True, enable_aqe=True)

print("=== Production Databricks Performance Parameters ===")
for k, v in configs.items():
    print(f"{k:60s} = {v}")

print("
=== Delta Z-Ordering Command ===")
print(optimizer.generate_zorder_optimize_sql("dev_lakehouse.bronze_landing.raw_logs", ["event_date", "user_id"]))

--- 
## Module 5: MLflow ETL Pipeline Metric Tracking

In [ ]:
from src.mlflow_integration import MLflowTracker

tracker = MLflowTracker("/Shared/fundamentals_notebook_run")
run_id = tracker.start_pipeline_run("fundamentals_notebook_execution")

tracker.log_pipeline_params({
    "spark_version": "14.3.x",
    "target_table": "dev_lakehouse.bronze_landing.raw_logs"
})

tracker.log_pipeline_metrics({
    "ingested_records": 250000.0,
    "validation_pass_rate": 0.998,
    "latency_sec": 14.5
})

tracker.end_pipeline_run()
print(f"Logged MLflow run successfully with ID: {run_id}")